# Phyllome CEA — Crop Growth & Financial Viability Pipeline
**Author:** Diego Fernández G. — Master by Research, University of Sydney  
**Data:** Phyllome vertical farm, Jun 2024 – Dec 2025  
**Goal:** Calibrate crop growth models and bridge them to a financial viability
framework for Controlled-Environment Agriculture (CEA) evaluation.

---
## Contents
1. Setup & data loading
2. Physical coordinate system & sensor positions
3. Microclimate assignment (anisotropic IDW interpolation)
4. Empirical yield models (OLS + Random Forest with LOSO-CV)
5. Biomass growth curves (Gompertz)
6. Mechanistic growth model (Vanthoor, simplified)
7. Summary of calibrated parameters
8. Financial bridge module — R/m²/year viability analysis
9. Operating window optimisation — yield per kWh & sensitivity
10. Integrated summary


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import curve_fit, minimize
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

BASE    = "data/"
UPLOADS = "data/raw/"

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PPFD_COLORS  = {180.0: '#4C9BE8', 310.0: '#F4A020'}
COL_VIABLE   = '#2E7D32'
COL_MARGINAL = '#E9C46A'
COL_LOSS     = '#E63946'

print('Libraries loaded ✓')


In [ ]:
air   = pd.read_csv(f"{BASE}air_clean.csv",    parse_dates=['timestamp'])
water = pd.read_csv(f"{BASE}water_clean.csv",  parse_dates=['timestamp'])
prod  = pd.read_csv(f"{BASE}production_clean.csv",
                    parse_dates=['growth_start','harvest_date'])
mc    = pd.read_csv(f"{BASE}tray_microclimate.csv",
                    parse_dates=['growth_start','harvest_date'])

print(f"Air sensor data   : {len(air):,} rows | {air['deviceId'].nunique()} sensors")
print(f"Water sensor data : {len(water):,} rows")
print(f"Production data   : {len(prod):,} trays | {prod['Season'].nunique()} seasons")
print(f"Tray microclimate : {len(mc):,} valid trays")
print()
print("Yield summary (g/tray):")
print(mc['yield_g'].describe().round(1).to_string())


## 2. Physical Coordinate System & Sensor Positions

- **4 columns**, **12 shelves**, **18 rows** per column
- Tray size: 1 155 × 1 155 mm | Row pitch: 1 175 mm | Shelf pitch: 500 mm
- 9 sensors in columns 1 and 3 only
- Column 4: −7 pp humidity bias correction applied
- Sensor c44c (col 1, shelf 1): CO₂ removed (malfunction)


In [ ]:
COL_X   = {1: 0, 2: 20, 3: 1160, 4: 1180}
ROW_Y   = lambda r: (r - 1) * 1175
SHELF_Z = lambda s: 700 + (s - 1) * 500

SENSORS = [
    ('c44cba6d9fe8', 1,  1,  3, False),
    ('484cba6d9fe8', 1,  8,  3, True),
    ('104dba6d9fe8', 1, 13,  3, True),
    ('884cba6d9fe8', 3,  2, 18, True),
    ('744cba6d9fe8', 3,  3,  3, True),
    ('bc4cba6d9fe8', 3,  8,  3, True),
    ('6c4cba6d9fe8', 3,  8, 18, True),
    ('0c4dba6d9fe8', 3, 13,  3, True),
    ('f04cba6d9fe8', 3, 13, 18, True),
]

sensor_df = pd.DataFrame(SENSORS, columns=['deviceId','col','shelf','row','has_co2'])
sensor_df['x_mm'] = sensor_df['col'].map(COL_X)
sensor_df['y_mm'] = sensor_df['row'].apply(ROW_Y)
sensor_df['z_mm'] = sensor_df['shelf'].apply(SHELF_Z)
sensor_df


## 3. Microclimate Assignment — Anisotropic IDW

$$d_{eff} = \sqrt{\Delta x^2 + \Delta y^2 + (\alpha \cdot \Delta z)^2}, \quad \alpha=3$$
$$w_i = \frac{1/d_{eff,i}^{2}}{\sum_j 1/d_{eff,j}^{2}}$$


In [ ]:
ALPHA_Z       = 3.0
IDW_POWER     = 2
COL4_H_OFFSET = -7.0

def compute_idw_weights(tray_col, tray_shelf, tray_row, variable='thv'):
    tx, ty, tz = COL_X[tray_col], ROW_Y(tray_row), SHELF_Z(tray_shelf)
    weights = {}
    for _, s in sensor_df.iterrows():
        if variable == 'co2' and not s['has_co2']:
            continue
        dx, dy = tx - s['x_mm'], ty - s['y_mm']
        dz = ALPHA_Z * (tz - s['z_mm'])
        dist = np.sqrt(dx**2 + dy**2 + dz**2)
        if dist == 0:
            return {s['deviceId']: 1.0}
        weights[s['deviceId']] = 1.0 / dist**IDW_POWER
    total = sum(weights.values())
    return {k: v/total for k, v in weights.items()}

valid = mc.copy()
print("Microclimate coverage:")
for col in ['air_temp_c','air_humidity_pct','co2_ppm','vpd_kPa','ec_mS','ph','water_temp_c']:
    print(f"  {col:<22}: {valid[col].notna().mean()*100:.1f}%")
print()
valid[['air_temp_c','vpd_kPa','co2_ppm','ec_mS','ph','water_temp_c']].describe().round(2)


## 4. Empirical Yield Models

| Model | Validation | Key result |
|---|---|---|
| OLS regression | R² full dataset | 0.61 |
| Random Forest | LOSO-CV | ~0.50 |

DLI is the dominant predictor (RF MDI = 0.37). PPFD 180→310 µmol/m²/s causes ~2.5× yield increase.


In [ ]:
# OLS with VIF diagnostics
features_ols = ['dli','growth_days','vpd_kPa','air_temp_c','co2_ppm',
                'air_humidity_pct','ec_mS','ph','water_temp_c','Shelf','Column']

sub   = valid[['yield_g'] + features_ols].dropna()
X_raw = sub[features_ols]
X_std = sm.add_constant((X_raw - X_raw.mean()) / X_raw.std())
ols   = sm.OLS(sub['yield_g'], X_std).fit()

print(f"OLS  R²={ols.rsquared:.3f}  adj-R²={ols.rsquared_adj:.3f}  RMSE={np.sqrt(ols.mse_resid):.1f}g  n={len(sub)}")
print()

res_df = pd.DataFrame({'coef': ols.params, 'p': ols.pvalues}).drop('const')
res_df['sig'] = res_df['p'].apply(lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else '')))
print(res_df.sort_values('coef', key=abs, ascending=False).round(4).to_string())
print()

vif_df = pd.DataFrame({'feature': features_ols,
    'VIF': [variance_inflation_factor(X_raw.values, i) for i in range(X_raw.shape[1])]
}).sort_values('VIF', ascending=False)
print("VIF:\n", vif_df.round(2).to_string(index=False))


In [ ]:
# OLS residual diagnostics (4-panel)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
resid, fitted = ols.resid, ols.fittedvalues

axes[0,0].scatter(fitted, resid, alpha=0.15, s=4, color='#374151')
axes[0,0].axhline(0, color='k', lw=1)
axes[0,0].set(xlabel='Fitted (g)', ylabel='Residuals (g)', title='Residuals vs Fitted')

(osm, osr), (slope, intercept, _) = stats.probplot(resid, dist='norm')
axes[0,1].scatter(osm, osr, alpha=0.3, s=6, color='#374151')
axes[0,1].plot([osm.min(),osm.max()],[slope*osm.min()+intercept,slope*osm.max()+intercept],'r-',lw=1.5)
axes[0,1].set(xlabel='Theoretical quantiles', ylabel='Sample quantiles',
              title=f'Normal Q-Q  (SW p={stats.shapiro(resid[:5000])[1]:.4f})')

axes[1,0].scatter(fitted, np.sqrt(np.abs(resid)), alpha=0.15, s=4, color='#374151')
axes[1,0].set(xlabel='Fitted (g)', ylabel='\u221a|Residuals|', title='Scale-Location')

axes[1,1].hist(resid, bins=60, color='#4C9BE8', edgecolor='white', linewidth=0.3)
axes[1,1].set(xlabel='Residual (g)', ylabel='Count',
              title=f'Residual distribution  (skew={stats.skew(resid):.2f})')

plt.suptitle('OLS Residual Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Random Forest — Leave-One-Season-Out CV
features_rf = ['dli','growth_days','vpd_kPa','air_temp_c','co2_ppm',
               'air_humidity_pct','ec_mS','ph','water_temp_c','Shelf','Column','Row']

df_rf     = valid[['Season','yield_g','ppfd'] + features_rf].dropna()
seasons   = sorted(df_rf['Season'].unique())
oof_pred  = np.full(len(df_rf), np.nan)
oof_idx   = {}

print(f"LOSO-CV over {len(seasons)} seasons...")
for i, s_out in enumerate(seasons):
    tr_idx = df_rf.index[df_rf['Season'] != s_out]
    te_idx = df_rf.index[df_rf['Season'] == s_out]
    rf_cv  = RandomForestRegressor(200, max_features='sqrt', min_samples_leaf=15, random_state=42, n_jobs=-1)
    rf_cv.fit(df_rf.loc[tr_idx, features_rf], df_rf.loc[tr_idx, 'yield_g'])
    oof_pred[df_rf.index.get_indexer(te_idx)] = rf_cv.predict(df_rf.loc[te_idx, features_rf])
    oof_idx[s_out] = r2_score(df_rf.loc[te_idx,'yield_g'], rf_cv.predict(df_rf.loc[te_idx, features_rf]))
    print(f"  [{i+1:2d}/{len(seasons)}] {s_out}  n={len(te_idx):4d}  R²={oof_idx[s_out]:.3f}")

rf_full = RandomForestRegressor(300, max_features='sqrt', min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_full.fit(df_rf[features_rf], df_rf['yield_g'])

r2_loso   = r2_score(df_rf['yield_g'], oof_pred)
rmse_loso = np.sqrt(mean_squared_error(df_rf['yield_g'], oof_pred))
print(f"\nLOSO R²={r2_loso:.3f}  RMSE={rmse_loso:.1f}g  n={len(df_rf)}")


In [ ]:
# RF diagnostics: MDI + Permutation importance + OOF scatter
imp_mdi  = pd.Series(rf_full.feature_importances_, index=features_rf)
perm     = permutation_importance(rf_full, df_rf[features_rf], df_rf['yield_g'],
                                   n_repeats=10, random_state=42, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=features_rf)
imp_std  = pd.Series(perm.importances_std,  index=features_rf)
order    = imp_mdi.sort_values(ascending=True).index

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].barh(range(len(order)), imp_mdi[order], height=0.6, color='#4C9BE8', alpha=0.85)
axes[0].set_yticks(range(len(order))); axes[0].set_yticklabels(order, fontsize=8)
axes[0].set(xlabel='MDI', title='RF — MDI Importance')

axes[1].barh(range(len(order)), imp_perm[order], xerr=imp_std[order], height=0.6,
             color='#F4A020', alpha=0.85, ecolor='#374151', capsize=2)
axes[1].set_yticks(range(len(order))); axes[1].set_yticklabels(order, fontsize=8)
axes[1].set(xlabel='Permutation (±SD)', title='RF — Permutation Importance')

for ppfd, grp in df_rf.groupby('ppfd'):
    idx = grp.index
    axes[2].scatter(grp['yield_g'], oof_pred[df_rf.index.get_indexer(idx)],
                    color=PPFD_COLORS[ppfd], alpha=0.15, s=5, label=f'PPFD {int(ppfd)}')
axes[2].plot([0,3400],[0,3400],'k--',lw=1)
axes[2].set(xlabel='Actual (g)', ylabel='OOF predicted (g)',
            title=f'LOSO-CV  R²={r2_loso:.3f} | RMSE={rmse_loso:.0f}g')
axes[2].legend(markerscale=3)

plt.suptitle('Random Forest — Full Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Biomass Growth Curves (Gompertz)

$$W(t) = A \cdot \exp(-\exp(-k(t-t_i)))$$

Mean R²=0.998 | PPFD 310 → k̄ 30% higher than PPFD 180 | DLI→k correlation r=+0.38


In [ ]:
avg_raw = pd.read_csv(f"{UPLOADS}Average data per season - Average data per season.csv")
bm_cols = ['Weight D5 (g)','Weight D10 (g)','Weight D15 (g)','Weight D20 (g)']
days_m  = [5, 10, 15, 20]

def gompertz(t, A, k, ti):
    return A * np.exp(-np.exp(-k * (t - ti)))

gompertz_results = []
for _, row in avg_raw.iterrows():
    pts = [(d, row[f'Weight D{d} (g)']) for d in days_m
           if pd.notna(row[f'Weight D{d} (g)']) and row[f'Weight D{d} (g)'] > 0]
    if len(pts) < 3: continue
    t_arr, w_arr = np.array([p[0] for p in pts], float), np.array([p[1] for p in pts], float)
    try:
        popt, pcov = curve_fit(gompertz, t_arr, w_arr, p0=[w_arr.max()*1.5, 0.3, 12.0],
                               bounds=([0,0.01,0],[100,5,40]), maxfev=5000)
        r2 = 1 - np.sum((w_arr-gompertz(t_arr,*popt))**2)/np.sum((w_arr-w_arr.mean())**2)
        gompertz_results.append({'Season':row['Harvest'],'PPFD':row['PPFD (µmol/m²/s)'],
            'DLI':row['DLI (mol/m²/day)'],'A':popt[0],'k':popt[1],'ti':popt[2],
            'k_se':np.sqrt(np.diag(pcov))[1],'r2':r2,'Yield_g':row['Avg Yield/Tray (g)']})
    except: pass

gom_df = pd.DataFrame(gompertz_results)
print(f"Fitted: {len(gom_df)} seasons | Mean R²={gom_df['r2'].mean():.4f}")
for ppfd, grp in gom_df.groupby('PPFD'):
    k_vals = grp['k'].values
    ci = stats.t.interval(0.95, len(k_vals)-1, loc=k_vals.mean(), scale=stats.sem(k_vals))
    print(f"PPFD {ppfd:>4.0f}: k={k_vals.mean():.4f} [95%CI {ci[0]:.4f}-{ci[1]:.4f}]")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ppfd, grp in gom_df.groupby('PPFD'):
    axes[0].scatter(grp['DLI'], grp['k'], color=PPFD_COLORS[ppfd], alpha=0.8, s=60, label=f'PPFD {int(ppfd)}')
p_lin = np.polyfit(gom_df['DLI'], gom_df['k'], 1)
xr = np.linspace(gom_df['DLI'].min(), gom_df['DLI'].max(), 100)
r_val, p_val = stats.pearsonr(gom_df['DLI'], gom_df['k'])
axes[0].plot(xr, np.polyval(p_lin,xr), 'k--', lw=1.5, label=f'r={r_val:.2f}, p={p_val:.3f}')
axes[0].set(xlabel='DLI', ylabel='k (day⁻¹)', title='Growth rate k vs DLI'); axes[0].legend(fontsize=8)

axes[1].hist(gom_df[gom_df['PPFD']==180]['k'], bins=15, color='#4C9BE8', alpha=0.65, label='PPFD 180')
axes[1].hist(gom_df[gom_df['PPFD']==310]['k'], bins=10, color='#F4A020', alpha=0.65, label='PPFD 310')
axes[1].set(xlabel='k (day⁻¹)', ylabel='Count', title='Distribution of k'); axes[1].legend()

for ppfd, grp in gom_df.groupby('PPFD'):
    axes[2].scatter(grp['A'], grp['Yield_g'], color=PPFD_COLORS[ppfd], alpha=0.8, s=50, label=f'PPFD {int(ppfd)}')
axes[2].set(xlabel='Asymptote A (g)', ylabel='Observed yield (g)', title='A vs Harvest Yield'); axes[2].legend()

plt.suptitle('Gompertz Growth Model', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Mechanistic Growth Model (Vanthoor, simplified)

$$\frac{dW}{dt} = \varepsilon_{LUE} \cdot I_{PAR} \cdot (1-e^{-K\cdot SLA\cdot W}) \cdot L_h - R_m \cdot W \cdot Q_{10}^{(T-20)/10}$$

| Parameter | Value | Source |
|---|---|---|
| ε_LUE | 1.673 g DM/mol PAR | grid-search calibration |
| K | 0.80 | literature |
| SLA | 0.028 m²/g | literature |
| Rm | 0.012 day⁻¹ | literature |
| DM% | 6% | assumed |


In [ ]:
TRAY_AREA = 1.155**2; DM_FRACTION = 0.06; EPS_LUE = 1.673; K_EXT = 0.80; SLA = 0.028; RM = 0.012

def vanthoor_simulate(row, eps=EPS_LUE, dm=DM_FRACTION):
    if pd.isna(row.get('Weight D5 (g)')) or row['Weight D5 (g)'] <= 0: return {}
    plants = row.get('plants_per_tray', 900)
    DW0    = row['Weight D5 (g)'] * dm * plants / TRAY_AREA
    T, PPFD, lh = row['Avg Temp (°C)'], row['PPFD (µmol/m²/s)'], row['Light h/day']
    Q10f = 2.0**((T-20)/10); PAR_h = PPFD * 1e-6 * 3600
    DW, out = DW0, {5: DW0}
    for d in range(6, int(row['Growing Days'])+1):
        dDW = eps * PAR_h * (1-np.exp(-K_EXT*SLA*DW)) * lh - RM * DW * Q10f
        DW  = max(DW + dDW, 0.0); out[d] = DW
    return out

ppts  = prod.groupby('Season')['plants_per_tray'].first().reset_index()
ppts.columns = ['Harvest','plants_per_tray']
avg2  = avg_raw.merge(ppts, on='Harvest', how='left')
avg2['plants_per_tray'] = avg2['plants_per_tray'].fillna(900)

y_obs, y_pred_v = [], []
for _, row in avg2.iterrows():
    for d in [10,15,20]:
        fw = row.get(f'Weight D{d} (g)')
        if pd.notna(fw) and fw>0 and pd.notna(row.get('Weight D5 (g)')) and row['Weight D5 (g)']>0:
            dw_obs = fw * DM_FRACTION * row['plants_per_tray'] / TRAY_AREA
            sim = vanthoor_simulate(row)
            if d in sim: y_obs.append(dw_obs); y_pred_v.append(sim[d])

y_obs = np.array(y_obs); y_pred_v = np.array(y_pred_v)
print(f"Vanthoor  R²={r2_score(y_obs,y_pred_v):.3f}  MAPE={np.mean(np.abs(y_obs-y_pred_v)/y_obs)*100:.1f}%  n={len(y_obs)}")


In [ ]:
# eps_LUE grid search + diagnostics
eps_grid = np.linspace(0.5, 3.5, 120); mape_grid = []
for eps in eps_grid:
    yo, yp = [], []
    for _, row in avg2.iterrows():
        for d in [10,15,20]:
            fw = row.get(f'Weight D{d} (g)')
            if pd.notna(fw) and fw>0 and pd.notna(row.get('Weight D5 (g)')) and row['Weight D5 (g)']>0:
                dw_o = fw*DM_FRACTION*row['plants_per_tray']/TRAY_AREA
                sim  = vanthoor_simulate(row, eps=eps)
                if d in sim: yo.append(dw_o); yp.append(sim[d])
    yo=np.array(yo); yp=np.array(yp)
    mape_grid.append(np.mean(np.abs(yo-yp)/yo)*100)
best_eps = eps_grid[np.argmin(mape_grid)]
print(f"Best eps_LUE={best_eps:.3f}  MAPE={min(mape_grid):.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].plot(eps_grid, mape_grid, color='#4C9BE8', lw=2)
axes[0].axvline(best_eps, color=COL_LOSS, lw=1.5, ls='--', label=f'eps={best_eps:.3f}')
axes[0].set(xlabel='eps_LUE', ylabel='MAPE (%)', title='Grid search'); axes[0].legend()

for _, row in avg2.iterrows():
    ppfd=row['PPFD (µmol/m²/s)']; sim=vanthoor_simulate(row)
    for d in [10,15,20]:
        fw=row.get(f'Weight D{d} (g)')
        if pd.notna(fw) and fw>0 and d in sim:
            axes[1].scatter(fw*DM_FRACTION*row['plants_per_tray']/TRAY_AREA, sim[d],
                            color=PPFD_COLORS.get(ppfd,'gray'), alpha=0.55, s=20)
axes[1].plot([0,max(y_obs.max(),y_pred_v.max())*1.05]*2,[0,max(y_obs.max(),y_pred_v.max())*1.05],'k--',lw=1)
axes[1].set(xlabel='Observed DW/m²', ylabel='Predicted DW/m²',
            title=f'Predicted vs Observed  R²={r2_score(y_obs,y_pred_v):.3f}')

sel = avg2[avg2[bm_cols].notna().sum(axis=1)>=3].head(8)
for _, row in sel.iterrows():
    if pd.isna(row.get('Weight D5 (g)')) or row['Weight D5 (g)']<=0: continue
    sim=vanthoor_simulate(row); c=PPFD_COLORS.get(row['PPFD (µmol/m²/s)'],'gray')
    axes[2].plot(list(sim.keys()),list(sim.values()),color=c,alpha=0.6,lw=1.8)
    obs={d:row[f'Weight D{d} (g)']*DM_FRACTION*row['plants_per_tray']/TRAY_AREA
         for d in days_m if pd.notna(row.get(f'Weight D{d} (g)')) and row[f'Weight D{d} (g)']>0}
    axes[2].scatter(list(obs.keys()),list(obs.values()),color=c,s=45,zorder=5)
axes[2].set(xlabel='Days', ylabel='DW/m² (g)', title='Growth trajectories')

plt.suptitle('Vanthoor Mechanistic Model', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 7. Summary of Calibrated Parameters

| Model | Metric | Value |
|---|---|---|
| OLS | R² | 0.61 |
| Random Forest (LOSO-CV) | CV R² | ~0.50 |
| Gompertz | Mean R² | 0.998 |
| Vanthoor | MAPE | 31.3% |

---


## 8. Financial Bridge Module — R/m²/year Viability Analysis

$$R/m^2/yr = Y \cdot P - OPEX - CAPEX_{amort}$$

$$Y^* = \frac{OPEX + CAPEX_{amort}}{P}$$

| Parameter | Value | Source |
|---|---|---|
| OPEX | 120 USD/m²/yr | Al-Chalabi 2015; Benke & Tomkins 2017 |
| CAPEX amort | 45 USD/m²/yr | Graamans et al. 2018 |
| Baseline P | 8 USD/kg | CEA spinach literature |


In [ ]:
TRAY_AREA_M2 = 1.155**2
OPEX_M2_YEAR = 120.0
CAPEX_AMORT  = 45.0
TOTAL_COST   = OPEX_M2_YEAR + CAPEX_AMORT
P_BASE       = 8.0

mean_gd     = mc['growth_days'].mean()
cycles_year = 365.0 / mean_gd

mc['yield_kg_cycle_m2'] = mc['yield_g'] / 1000 / TRAY_AREA_M2
mc['Y_kg_m2_year']      = mc['yield_kg_cycle_m2'] * cycles_year
mc['R_m2_year']         = mc['Y_kg_m2_year'] * P_BASE - TOTAL_COST
Y_star = TOTAL_COST / P_BASE

print(f"Growth days: {mean_gd:.1f}d  →  {cycles_year:.1f} cycles/year")
print(f"Break-even Y* = {Y_star:.2f} kg/m²/yr  (P=${P_BASE}/kg, cost=${TOTAL_COST}/m²/yr)")
print(f"Mean Y = {mc['Y_kg_m2_year'].mean():.1f} kg/m²/yr")
print(f"Mean R = {mc['R_m2_year'].mean():.1f} USD/m²/yr")
print(f"Fraction viable = {(mc['R_m2_year']>0).mean()*100:.1f}%")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Y distribution vs Y*
ax = axes[0,0]
for ppfd, grp in mc.groupby('ppfd'):
    ax.hist(grp['Y_kg_m2_year'], bins=40, alpha=0.6, color=PPFD_COLORS[ppfd],
            label=f'PPFD {int(ppfd)}', density=True)
ax.axvline(Y_star, color=COL_LOSS, lw=2, ls='--', label=f'Y*={Y_star:.1f} kg/m²/yr')
ax.set(xlabel='Y (kg/m²/yr)', ylabel='Density', title='Yield distribution vs break-even'); ax.legend(fontsize=8)

# R distribution
ax = axes[0,1]
r_vals = mc['R_m2_year'].dropna()
ax.hist(r_vals[r_vals<0], bins=40, color=COL_LOSS, alpha=0.7, label='Loss')
ax.hist(r_vals[r_vals>=0], bins=40, color=COL_VIABLE, alpha=0.7, label='Viable')
ax.axvline(0, color='k', lw=1.5, ls='--')
ax.set(xlabel='R/m²/yr (USD)', ylabel='Count', title=f'R distribution (P=${P_BASE}/kg)'); ax.legend()

# Break-even chart
ax = axes[1,0]
P_range = np.linspace(2, 24, 200)
for ppfd, grp in mc.groupby('ppfd'):
    Y_med, Y_q25, Y_q75 = grp['Y_kg_m2_year'].median(), grp['Y_kg_m2_year'].quantile(0.25), grp['Y_kg_m2_year'].quantile(0.75)
    c = PPFD_COLORS[ppfd]
    ax.plot(P_range, Y_med*P_range-TOTAL_COST, color=c, lw=2.5, label=f'PPFD {int(ppfd)}')
    ax.fill_between(P_range, Y_q25*P_range-TOTAL_COST, Y_q75*P_range-TOTAL_COST, color=c, alpha=0.15)
ax.axhline(0, color='k', lw=1, ls='--'); ax.axvline(P_BASE, color='#374151', lw=1, ls=':')
ax.set(xlabel='P (USD/kg)', ylabel='R/m²/yr (USD)', title='Break-even chart'); ax.legend(fontsize=8)

# Viability heatmap
ax = axes[1,1]
P_grid, O_grid = np.linspace(4,20,80), np.linspace(60,200,80)
PP, OO = np.meshgrid(P_grid, O_grid)
RR = mc['Y_kg_m2_year'].median() * PP - OO - CAPEX_AMORT
contf = ax.contourf(PP, OO, RR, levels=20, cmap='RdYlGn')
ax.contour(PP, OO, RR, levels=[0], colors='k', linewidths=2)
plt.colorbar(contf, ax=ax, label='R/m²/yr (USD)')
ax.scatter([P_BASE],[OPEX_M2_YEAR], color='white', s=120, zorder=5, edgecolors='k', label='Baseline')
ax.set(xlabel='P (USD/kg)', ylabel='OPEX (USD/m²/yr)', title='Viability heatmap'); ax.legend(fontsize=8)

plt.suptitle('Financial Bridge — R/m²/year Viability', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Monte Carlo risk analysis
np.random.seed(42); N_MC = 50_000
Y_obs = mc['Y_kg_m2_year'].dropna()
Y_mc  = np.random.lognormal(np.log(Y_obs).mean(), np.log(Y_obs).std(), N_MC)

P_scenarios = {'Pessimistic (P=5)':5.0, 'Baseline (P=8)':8.0, 'Optimistic (P=14)':14.0}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_mc = {}
for label, P_val in P_scenarios.items():
    P_mc = P_val * np.random.normal(1.0, 0.10, N_MC)
    R_mc = Y_mc * P_mc - TOTAL_COST
    pv   = (R_mc > 0).mean() * 100
    results_mc[label] = {'P':P_val,'R_p10':np.percentile(R_mc,10),'R_p50':np.percentile(R_mc,50),
                          'R_p90':np.percentile(R_mc,90),'prob_viable':pv}
    c = COL_VIABLE if pv>60 else (COL_MARGINAL if pv>30 else COL_LOSS)
    axes[0].hist(R_mc, bins=80, density=True, alpha=0.55, color=c, label=f'{label}: {pv:.0f}%')
axes[0].axvline(0, color='k', lw=1.5, ls='--')
axes[0].set(xlabel='R/m²/yr (USD)', ylabel='Density', title='Monte Carlo (N=50k)')
axes[0].legend(fontsize=8)

Y_base = Y_obs.median(); R_base = Y_base*P_BASE - TOTAL_COST
params = {'Y (+10%)':(Y_base*1.1*P_BASE-TOTAL_COST)-R_base, 'Y (-10%)':(Y_base*0.9*P_BASE-TOTAL_COST)-R_base,
          'P (+10%)':(Y_base*P_BASE*1.1-TOTAL_COST)-R_base, 'P (-10%)':(Y_base*P_BASE*0.9-TOTAL_COST)-R_base,
          'OPEX (+10%)':(Y_base*P_BASE-TOTAL_COST*1.1)-R_base, 'OPEX (-10%)':(Y_base*P_BASE-TOTAL_COST*0.9)-R_base,
          'Cycles (+1)':((Y_base/cycles_year)*(cycles_year+1)*P_BASE-TOTAL_COST)-R_base,
          'Cycles (-1)':((Y_base/cycles_year)*(cycles_year-1)*P_BASE-TOTAL_COST)-R_base}
deltas = list(params.values()); labels_t = list(params.keys())
axes[1].barh(range(len(labels_t)), deltas, color=[COL_VIABLE if d>=0 else COL_LOSS for d in deltas], height=0.6)
axes[1].axvline(0, color='k', lw=1)
axes[1].set_yticks(range(len(labels_t))); axes[1].set_yticklabels(labels_t, fontsize=8)
axes[1].set(xlabel='Delta R/m²/yr', title=f'Tornado (base R={R_base:.1f})')

plt.suptitle('Financial Risk Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(pd.DataFrame(results_mc).T[['P','R_p10','R_p50','R_p90','prob_viable']].round(2).to_string())


## 9. Operating Window Optimisation

Identify conditions that maximise yield per kWh and reach Y* at minimum cost.


In [ ]:
mc_fin = mc.dropna(subset=['watts_per_tray','yield_g','dli','vpd_kPa','air_temp_c']).copy()
mc_fin['kwh_per_cycle'] = mc_fin['watts_per_tray'] * mc_fin['growth_days'] * 16 / 1000
mc_fin['yield_per_kwh'] = mc_fin['yield_g'] / mc_fin['kwh_per_cycle']
mc_fin['R_m2_year']     = mc_fin['Y_kg_m2_year'] * P_BASE - TOTAL_COST

print("Energy efficiency by PPFD:")
for ppfd, grp in mc_fin.groupby('ppfd'):
    print(f"  PPFD {ppfd:.0f}: {grp['yield_per_kwh'].median():.1f} g/kWh "
          f"(IQR {grp['yield_per_kwh'].quantile(0.25):.0f}-{grp['yield_per_kwh'].quantile(0.75):.0f})")

mc_fin['dli_bin'] = pd.cut(mc_fin['dli'], bins=np.arange(0,25,2.5),
                            labels=[f'{v:.1f}' for v in np.arange(1.25,22.5,2.5)])
dli_agg = mc_fin.groupby('dli_bin', observed=True)[['yield_g','yield_per_kwh']].agg(['mean','sem'])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].bar(range(len(dli_agg)), dli_agg['yield_g']['mean'], yerr=dli_agg['yield_g']['sem'],
            color='#4C9BE8', alpha=0.8, capsize=3)
axes[0].axhline(Y_star*TRAY_AREA_M2*1000/cycles_year, color=COL_LOSS, ls='--', lw=1.5, label='Y* per cycle')
axes[0].set_xticks(range(len(dli_agg))); axes[0].set_xticklabels(dli_agg.index, rotation=45, fontsize=8)
axes[0].set(xlabel='DLI bin', ylabel='Mean yield (g)', title='Yield vs DLI (±SEM)'); axes[0].legend(fontsize=8)

axes[1].bar(range(len(dli_agg)), dli_agg['yield_per_kwh']['mean'], yerr=dli_agg['yield_per_kwh']['sem'],
            color='#E9C46A', alpha=0.85, capsize=3)
axes[1].set_xticks(range(len(dli_agg))); axes[1].set_xticklabels(dli_agg.index, rotation=45, fontsize=8)
axes[1].set(xlabel='DLI bin', ylabel='g/kWh', title='Energy efficiency vs DLI')

sc = axes[2].scatter(mc_fin['dli'], mc_fin['R_m2_year'], c=mc_fin['air_temp_c'],
                      cmap='RdYlGn_r', alpha=0.2, s=5, vmin=18, vmax=26)
plt.colorbar(sc, ax=axes[2], label='Temp (°C)')
axes[2].axhline(0, color='k', lw=1.5, ls='--', label='Break-even')
axes[2].set(xlabel='DLI', ylabel='R/m²/yr (USD)', title=f'DLI vs R  (P=${P_BASE}/kg)'); axes[2].legend(fontsize=8)

plt.suptitle('Operating Window Optimisation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 10. Integrated Summary

### Financial viability (literature benchmarks)
| Scenario | Y (kg/m²/yr) | R/m²/yr (USD) | P(viable) |
|---|---|---|---|
| All trays (mean) | ~14 | −52 | ~25% |
| PPFD 310 (mean) | ~26 | +43 | ~65% |
| Y* break-even (P=$8) | **20.6** | 0 | — |

### Key levers
1. **PPFD 310 µmol/m²/s** → most tractable path to cross Y*
2. **Price > $10.5/kg** → break-even at current mean yield
3. **OPEX < $95/m²/yr** → break-even at PPFD-310 yields
4. **DLI 15–22 mol/m²/day** → optimal energy efficiency

### To improve Vanthoor (MAPE < 15%)
1. Oven-dry ~10 plants (65°C, 48h) → measure actual DM fraction
2. Re-calibrate ε_LUE with measured DM%
3. Consider nutrient solution EC as a growth modifier

### Citation
```
Fernández G., D. (2025). Crop Growth and Financial Viability Pipeline for CEA.
University of Sydney Master by Research.
https://github.com/diegofg94/phyllome-crop-modelling
```
